# ENG 6010 Test 2, Q1: crime-scene zone classifier (Colab)

The Go2's front-camera frame is used to predict which of 5 crime-scene zones the dog is in (house 185):
`victim_area`, `bed_area`, `lounge_area`, `desk_entry_area` and `bathroom`.

The models are:
- **ResNet18** pretrained on ImageNet and fine-tuned;
- a **small CNN** trained from scratch;
- **colour-histogram + logistic-regression** baselines, in RGB and in HSV.

The data comes from the Q2 simulation and is split **by run**, so the test routes and the held-out scene are never seen in training.

Runtime: GPU recommended (Runtime > Change runtime type > T4/A100). Everything also runs on CPU, just more slowly.

In [ ]:
# 1. Get the code (the Test 2 branch) and the simulator submodule (needed only to regenerate data)
!git clone -q --branch claude/eng6010-test2-crime-scene-0rst3r https://github.com/CDJ7/Unitree-Dog-Claude-Dog-.git
%cd Unitree-Dog-Claude-Dog-/Test2
!pip -q install -r requirements.txt
import torch; print(torch.__version__, 'CUDA' if torch.cuda.is_available() else 'CPU')

## 2. Data

Choose **one** option:
- **A (recommended):** you have `data_full.zip` (a zip of `Test2/data/full/`) on Google Drive. Mount the drive and unzip it.
- **B:** regenerate the full dataset in Colab (36 runs, about 17k frames). This needs the MuJoCo simulator submodule and takes about 30-60 min.
- **C:** use only the committed sample run, for a quick pipeline check. The numbers are not meaningful.

In [ ]:
DATA_OPTION = 'A'   # 'A', 'B' or 'C'
import os
if DATA_OPTION == 'A':
    from google.colab import drive; drive.mount('/content/drive')
    !mkdir -p data && unzip -q -o /content/drive/MyDrive/data_full.zip -d data/
elif DATA_OPTION == 'B':
    !git -C .. submodule update --init --depth 1 Phase_2_Investigator/third_party/robot-unitree-go2-mujoco
    !apt-get -qq install -y libegl1 libosmesa6 > /dev/null
    os.environ['MUJOCO_GL'] = 'egl'
    !python q2_data_simulation/collect_dataset.py --workers 2
else:
    !mkdir -p data/full && cp -r data/sample/run_027 data/full/
!ls data/full | head; !ls data/full | wc -l

## 3. Baselines: colour histograms (RGB vs HSV) + logistic regression

In [ ]:
!python q1_supervised/baselines.py

## 4. Train ResNet18 (transfer learning) and the small CNN (from scratch)
Hyper-parameters are in `config/q1_train.yaml`; seeds are fixed. On Colab the ImageNet weights come straight from torchvision.

In [ ]:
!python q1_supervised/train.py --model resnet18

In [ ]:
!python q1_supervised/train.py --model smallcnn

## 5. Evaluate on the held-out test runs, Grad-CAM, demo

In [ ]:
!python q1_supervised/evaluate.py
!python q1_supervised/gradcam.py
!python q1_supervised/demo.py --run run_000

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/q1/comparison.md').read()))
for f in ['curves_resnet18.png', 'confusion_resnet18.png', 'confusion_hist_hsv.png', 'gradcam.png', 'demo_run_000.png', 'boundary.png']:
    display(Image(f'results/q1/{f}'))

## 6. Save the results to Drive (optional)

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/test2_results && cp -r results/q1 /content/drive/MyDrive/test2_results/